# MedTrace Federated Learning
## Privacy-Preserving Medical AI — Multi-Session Training with Verified Checkpointing

**Run All cells. If disconnected, just Run All again — resumes from last saved round.**

In [ ]:
# ================================================================
# CELL 1: SETUP — Install packages + Mount Drive with VERIFICATION
# ================================================================
!pip install -q transformers datasets peft accelerate torch

import torch, os
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

# ─── MOUNT DRIVE WITH VERIFICATION ───────────────────────
from google.colab import drive

def mount_and_verify_drive():
    """Mount Drive and verify it actually works by writing a test file."""
    if not os.path.exists('/content/drive/MyDrive'):
        print('Mounting Google Drive...')
        drive.mount('/content/drive')
    else:
        print('Drive path exists, verifying it works...')

    # Write a test file to confirm Drive is actually accessible
    test_path = '/content/drive/MyDrive/.medtrace_mount_test'
    try:
        with open(test_path, 'w') as f:
            f.write('medtrace_ok')
        with open(test_path, 'r') as f:
            content = f.read()
        os.remove(test_path)
        assert content == 'medtrace_ok', 'Test file content mismatch'
        print('Drive mounted and VERIFIED — writes are working correctly')
        return True
    except Exception as e:
        print(f'Drive verification FAILED: {e}')
        print('Forcing remount...')
        drive.mount('/content/drive', force_remount=True)
        # Try once more after remount
        try:
            with open(test_path, 'w') as f:
                f.write('medtrace_ok')
            os.remove(test_path)
            print('Drive remounted and VERIFIED')
            return True
        except Exception as e2:
            raise RuntimeError(f'Drive cannot be verified after remount: {e2}. Please check Drive access.')

mount_and_verify_drive()
print('\nSetup complete — Drive is ready for checkpointing')

In [ ]:
# ================================================================
# CELL 2: MEDTRACE FEDERATED LEARNING — COMPLETE TRAINING SCRIPT
# All code in ONE cell. Checkpoints verified after every save.
# ================================================================

import os, json, time, copy
import torch
import numpy as np
from collections import OrderedDict
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, PeftModel

# ─── CONFIG ───────────────────────────────────────────────
BASE_MODEL   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
NUM_HOSPITALS = 3
FL_ROUNDS    = 20
LOCAL_EPOCHS = 1
LORA_R       = 8
LORA_ALPHA   = 16
BATCH_SIZE   = 4
MAX_LENGTH   = 512
LR           = 2e-4
DP_EPSILON   = 8.0
DP_DELTA     = 1e-5
DP_MAX_GRAD_NORM = 1.0
SYSTEM_MSG   = 'You are MedTrace, a federated clinical reasoning system. Generate step-by-step auditable diagnostic reasoning chains.'

HOSPITAL_CONFIGS = {
    'hospital_A': {'name': 'Metro General (Cardiology)', 'kw': ['heart','cardiac','coronary','chest pain','myocardial','arrhythmia','angina','aortic'], 'samples': 1500},
    'hospital_B': {'name': 'Royal London (Neurology)',   'kw': ['brain','neuro','seizure','headache','stroke','cognitive','dementia','nerve'], 'samples': 1200},
    'hospital_C': {'name': 'AIIMS Delhi (Infectious)',   'kw': ['infection','fever','bacteria','virus','antibiotic','sepsis','tuberculosis','malaria'], 'samples': 1300},
}

CHECKPOINT_DIR = '/content/drive/MyDrive/MedTrace/fl_checkpoints'
SAVE_DIR       = '/content/drive/MyDrive/MedTrace/federated-global'
device         = 'cuda' if torch.cuda.is_available() else 'cpu'

# ─── CREATE AND VERIFY CHECKPOINT DIRECTORY ───────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# CRITICAL: Verify CHECKPOINT_DIR is actually in Google Drive (not local storage)
_verify_path = os.path.join(CHECKPOINT_DIR, '.dir_verify')
try:
    with open(_verify_path, 'w') as f:
        f.write('verified')
    assert os.path.exists(_verify_path), 'Verification file not found after write'
    os.remove(_verify_path)
    print(f'Checkpoint directory VERIFIED in Drive: {CHECKPOINT_DIR}')
except Exception as e:
    raise RuntimeError(
        f'CANNOT write to checkpoint directory: {e}\n'
        f'Drive may not be mounted. Run Cell 1 again before this cell.'
    )

print(f'Device: {device} | FL Rounds: {FL_ROUNDS} | Local Epochs: {LOCAL_EPOCHS}')

# ─── CHECKPOINT FUNCTIONS ─────────────────────────────────
def save_checkpoint(weights, round_num):
    """Save checkpoint to Drive and VERIFY the file exists with correct size."""
    path = os.path.join(CHECKPOINT_DIR, f'round_{round_num}.pt')
    marker = os.path.join(CHECKPOINT_DIR, 'last_round.txt')

    # Save weights
    torch.save(weights, path)

    # VERIFY the file actually exists in Drive
    if not os.path.exists(path):
        raise RuntimeError(
            f'CHECKPOINT SAVE FAILED: {path} not found after torch.save()\n'
            f'Drive may have disconnected. Check Drive mount.'
        )
    size_mb = os.path.getsize(path) / (1024 * 1024)
    if size_mb < 0.5:
        raise RuntimeError(
            f'CHECKPOINT TOO SMALL: {path} is only {size_mb:.2f}MB — save likely failed'
        )

    # Update the resume marker
    with open(marker, 'w') as f:
        f.write(str(round_num))

    # Verify marker
    with open(marker, 'r') as f:
        saved_val = f.read().strip()
    if saved_val != str(round_num):
        raise RuntimeError(f'Resume marker write failed: expected {round_num}, got {saved_val}')

    print(f'  CHECKPOINT SAVED & VERIFIED: round_{round_num}.pt ({size_mb:.1f} MB) → Drive')


def load_checkpoint():
    """Load last checkpoint from Drive. Returns (weights, last_round_num) or (None, -1)."""
    marker = os.path.join(CHECKPOINT_DIR, 'last_round.txt')
    if not os.path.exists(marker):
        print('  No checkpoint marker found — starting fresh')
        return None, -1

    with open(marker) as f:
        last = int(f.read().strip())

    path = os.path.join(CHECKPOINT_DIR, f'round_{last}.pt')
    if not os.path.exists(path):
        print(f'  Marker says round {last} but file not found: {path}')
        print('  Starting fresh (checkpoint file missing)')
        return None, -1

    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  Loading checkpoint: round_{last}.pt ({size_mb:.1f} MB)')
    w = torch.load(path, map_location='cpu', weights_only=False)
    print(f'  RESUMED from checkpoint — will start at Round {last + 2}/{FL_ROUNDS}')
    return w, last


# ─── HOSPITAL CLIENT ──────────────────────────────────────
class HospitalClient:
    def __init__(self, hid, config):
        self.hid          = hid
        self.name         = config['name']
        self.kw           = config['kw']
        self.num_samples  = config['samples']
        self.privacy_spent = 0.0
        print(f'  Hospital: {self.name}')

    def prepare_data(self, full_dataset, round_num):
        spec_idx, gen_idx = [], []
        for i, ex in enumerate(full_dataset):
            q = ex['question'].lower()
            if any(k in q for k in self.kw):
                spec_idx.append(i)
            else:
                gen_idx.append(i)
        seed  = abs(hash(self.hid) + round_num) % (2**32)
        rng   = np.random.RandomState(seed)
        n_spec = min(int(self.num_samples * 0.6), len(spec_idx))
        n_gen  = self.num_samples - n_spec
        chosen = []
        if n_spec > 0 and len(spec_idx) > 0:
            chosen.extend(rng.choice(spec_idx, n_spec, replace=True).tolist())
        if n_gen > 0 and len(gen_idx) > 0:
            chosen.extend(rng.choice(gen_idx, min(n_gen, len(gen_idx)), replace=True).tolist())
        seed2 = abs(hash(self.hid) + round_num + 42) % (2**32)
        rng2  = np.random.RandomState(seed2)
        rng2.shuffle(chosen)
        self.local_data = full_dataset.select(chosen[:self.num_samples])
        print(f'  {self.name}: {len(self.local_data)} samples ({n_spec} specialty)')

    def train_local(self, global_weights, tokenizer, round_num):
        print(f'  Training {self.name} (Round {round_num+1})...')
        t0 = time.time()
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
        lora_cfg = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
            target_modules=['q_proj','v_proj'], bias='none', task_type='CAUSAL_LM'
        )
        model = get_peft_model(base, lora_cfg)
        if global_weights is not None:
            model.load_state_dict(global_weights, strict=False)
        model.to(device)

        def tokenize(examples):
            prompts = [
                f'<|system|>\n{SYSTEM_MSG}</s>\n<|user|>\n{q}</s>\n<|assistant|>\n'
                for q in examples['question']
            ]
            enc = tokenizer(prompts, truncation=True, max_length=MAX_LENGTH, padding='max_length')
            enc['labels'] = enc['input_ids'].copy()
            return enc

        tok_data = self.local_data.map(tokenize, batched=True, remove_columns=self.local_data.column_names)
        tok_data.set_format('torch')
        out_dir = f'/content/fl_temp/{self.hid}/round_{round_num}'
        os.makedirs(out_dir, exist_ok=True)

        args = TrainingArguments(
            output_dir=out_dir,
            num_train_epochs=LOCAL_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            learning_rate=LR,
            warmup_steps=10,
            logging_steps=5,
            save_strategy='no',
            report_to='none',
            fp16=torch.cuda.is_available(),
            gradient_checkpointing=True,
            gradient_accumulation_steps=2,
            dataloader_pin_memory=False
        )
        trainer = Trainer(
            model=model, args=args, train_dataset=tok_data,
            data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
        )
        trainer.train()
        elapsed = time.time() - t0
        loss = trainer.state.log_history[-1].get('train_loss', 0) if trainer.state.log_history else 0

        # Extract LoRA weights only
        lora_w = OrderedDict()
        for name, p in model.named_parameters():
            if 'lora_' in name:
                lora_w[name] = p.detach().cpu().clone()

        # Differential privacy noise
        sigma = DP_MAX_GRAD_NORM * np.sqrt(2 * np.log(1.25 / DP_DELTA)) / DP_EPSILON
        for name in lora_w:
            norm = torch.norm(lora_w[name])
            if norm > DP_MAX_GRAD_NORM:
                lora_w[name] = lora_w[name] * (DP_MAX_GRAD_NORM / norm)
            lora_w[name] = lora_w[name] + torch.randn_like(lora_w[name]) * sigma

        self.privacy_spent += DP_EPSILON / FL_ROUNDS
        n_params = sum(p.numel() for p in lora_w.values())
        print(f'  DONE: Loss={loss:.4f} | Time={elapsed:.1f}s | Params={n_params:,}')
        print(f'  DP sigma={sigma:.4f} | Privacy spent: {self.privacy_spent:.2f}/{DP_EPSILON}')
        del model, base, trainer
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return lora_w, {'hospital': self.name, 'loss': loss, 'samples': len(self.local_data), 'time': elapsed}


# ─── FEDERATED SERVER ─────────────────────────────────────
class FederatedServer:
    def __init__(self):
        self.global_weights = None
        print('Aggregation server ready')

    def init_global_model(self):
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
        lora_cfg = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
            target_modules=['q_proj','v_proj'], bias='none', task_type='CAUSAL_LM'
        )
        model = get_peft_model(base, lora_cfg)
        self.global_weights = OrderedDict()
        for name, p in model.named_parameters():
            if 'lora_' in name:
                self.global_weights[name] = p.detach().cpu().clone()
        n = sum(p.numel() for p in self.global_weights.values())
        print(f'Global LoRA params: {n:,}')
        del model, base
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return self.global_weights

    def aggregate(self, client_updates, round_num):
        print(f'Aggregating Round {round_num+1} (FedAvg)...')
        wl, counts, names = [], [], []
        for hid, (w, m) in client_updates.items():
            wl.append(w); counts.append(m['samples']); names.append(m['hospital'])
        total = sum(counts)
        fracs = [c / total for c in counts]
        for n, f, c in zip(names, fracs, counts):
            print(f'  {n}: weight={f:.3f} ({c} samples)')
        agg = OrderedDict()
        for key in wl[0]:
            agg[key] = sum(fracs[i] * wl[i][key] for i in range(len(wl)))
        self.global_weights = agg
        div = sum(
            torch.norm(wl[i][k] - agg[k]).item()**2
            for k in agg for i in range(len(wl))
        ) / (len(agg) * len(wl))
        print(f'  Weight divergence: {div:.6f}')
        return self.global_weights


# ─── LOAD DATASET ─────────────────────────────────────────
print('Loading MedQA USMLE dataset...')
dataset = load_dataset('GBaker/MedQA-USMLE-4-options', split='train')
print(f'Total examples: {len(dataset)}')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ─── INIT + RESUME ────────────────────────────────────────
print('\n' + '='*70)
print('  MedTrace Federated Learning — Starting Up')
print('='*70)

server = FederatedServer()
global_w, last_round = load_checkpoint()
start_round = last_round + 1

if global_w is None:
    print('No checkpoint found — initializing fresh global model')
    global_w = server.init_global_model()
    start_round = 0
else:
    server.global_weights = global_w
    print(f'Resuming from Round {start_round + 1}/{FL_ROUNDS}')

if start_round >= FL_ROUNDS:
    print('All rounds already complete! Proceed to evaluation cell.')
else:
    hospitals = {}
    print('\nInitializing hospitals...')
    for hid, hcfg in HOSPITAL_CONFIGS.items():
        hospitals[hid] = HospitalClient(hid, hcfg)

    total_t0 = time.time()
    rounds_done = 0

    for r in range(start_round, FL_ROUNDS):
        print(f'\n{"="*70}')
        print(f'  ROUND {r+1}/{FL_ROUNDS}   (session rounds completed: {rounds_done})')
        print(f'{"="*70}')

        for hid, client in hospitals.items():
            client.prepare_data(dataset, r)

        updates = {}
        for hid, client in hospitals.items():
            w, m = client.train_local(global_w, tokenizer, r)
            updates[hid] = (w, m)

        global_w = server.aggregate(updates, r)

        # Save and VERIFY checkpoint in Drive
        save_checkpoint(global_w, r)
        rounds_done += 1

        elapsed_total = time.time() - total_t0
        avg_per_round = elapsed_total / rounds_done
        rounds_left   = FL_ROUNDS - (r + 1)
        eta_min       = rounds_left * avg_per_round / 60
        print(f'  Progress: {r+1}/{FL_ROUNDS} rounds | Avg/round: {avg_per_round/60:.1f}min | ETA: {eta_min:.0f}min')

    total_time = time.time() - total_t0
    print(f'\n{"="*70}')
    print(f'  TRAINING COMPLETE! Total time: {total_time/3600:.2f} hours')
    print(f'{"="*70}')

    # ─── SAVE FINAL MODEL TO DRIVE ────────────────────────
    print('\nSaving final federated model to Drive...')
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    lora_cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules=['q_proj','v_proj'], bias='none', task_type='CAUSAL_LM'
    )
    model = get_peft_model(base, lora_cfg)
    model.load_state_dict(global_w, strict=False)
    model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
    metadata = {
        'architecture': 'MedTrace Federated Learning',
        'base_model': BASE_MODEL,
        'hospitals': NUM_HOSPITALS,
        'rounds': FL_ROUNDS,
        'dp_epsilon': DP_EPSILON,
        'local_epochs': LOCAL_EPOCHS
    }
    with open(os.path.join(SAVE_DIR, 'fl_metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f'Final model saved to: {SAVE_DIR}')
    del model, base
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# ================================================================
# CELL 3: EVALUATE FEDERATED MODEL
# ================================================================
print('Loading federated model for evaluation...')
base  = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
model = PeftModel.from_pretrained(base, SAVE_DIR)
model.to(device)
model.eval()

questions = [
    'A 55-year-old woman has sudden left-sided weakness. CT shows no hemorrhage. Next step?',
    'A 30-year-old man has high fever, neck stiffness, photophobia. Diagnosis?',
    'A 65-year-old diabetic has crushing chest pain, elevated troponin. Management?',
]

for q in questions:
    prompt = f'<|system|>\n{SYSTEM_MSG}</s>\n<|user|>\n{q}</s>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=300, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    ans = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\nQ: {q}')
    print(f'A: {ans[:500]}')
    print('-' * 60)

del model, base
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# ================================================================
# CELL 4: INTERACTIVE MODE
# ================================================================
print('Loading model...')
base  = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
model = PeftModel.from_pretrained(base, SAVE_DIR)
model.to(device)
model.eval()
print('Ready! Type your question below.\n')

while True:
    question = input('Your question (quit to stop): ')
    if question.lower() == 'quit': break
    prompt = f'<|system|>\n{SYSTEM_MSG}</s>\n<|user|>\n{question}</s>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=400, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    ans = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n{"="*60}\n{ans}\n{"="*60}\n')